### **Silver Layer - Handling/ cleaning data**


## **Imporing libraries**

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### **Reading data**

Read bronze layer data

In [13]:
Silver_df = pd.read_csv("/content/sample_data/Bronze.csv")

### **Cleaning data**

1.Change data types

In [14]:
Silver_df["crash_date"] = pd.to_datetime(
    Silver_df["crash_date"],
    errors="coerce"
)

text_cols = [
    "traffic_control_device",
    "weather_condition",
    "lighting_condition",
    "first_crash_type",
    "trafficway_type",
    "alignment",
    "roadway_surface_cond",
    "road_defect",
    "crash_type",
    "intersection_related_i",
    "damage",
    "prim_contributory_cause",
    "most_severe_injury"
]

for col in text_cols:
    Silver_df[col] = Silver_df[col].astype(str)

/tmp/ipykernel_1423/3113058682.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  Silver_df["crash_date"] = pd.to_datetime(


2.Replace values

In [15]:
Silver_df["intersection_related_i"] = (
    Silver_df["intersection_related_i"]
    .replace({
        "Y": "YES",
        "N": "NO"
    })
)

3.Duplicate crash date

In [16]:
Silver_df["AM/PM"] = Silver_df["crash_date"]

4.Extract Am/Pm

In [17]:
Silver_df["AM/PM"] = (
    pd.to_datetime(Silver_df["AM/PM"])
    .dt.strftime("%p")
)


5.Convert crash_date to Date only

In [18]:
Silver_df["crash_date"] = Silver_df["crash_date"].dt.date

6.Rename columns

In [19]:
Silver_df = Silver_df.rename(columns={

    "intersection_related_i":
        "Intersection_accident?",

    "damage":
        "Cost_of_damage",

    "weather_condition":
        "Weather",

    "lighting_condition":
        "Lighting",

    "first_crash_type":
        "Collision Type",

    "alignment":
        "Road_alignment",

    "num_units":
        "Number_of_vehicles_involved",

    "injuries_total":
        "Total_injuries_num",

    "injuries_fatal":
        "Fatal_injuries_num",

    "injuries_reported_not_evident":
        "injuries_not_visibily_evident"

})

7.Adding Crash severity column

In [20]:
Silver_df["Crash_severity"] = np.select(
    [
        Silver_df["Fatal_injuries_num"] > 0,
        Silver_df["injuries_incapacitating"] > 0,
        Silver_df["injuries_non_incapacitating"] > 0,
        Silver_df["Total_injuries_num"] > 0
    ],
    [
        "Fatal",
        "Major",
        "Moderate",
        "Minor"
    ],
    default="No Injury"
)

In [21]:
Silver_df['Crash_severity'].sample(10)

,Crash_severity
138746,No Injury
46261,No Injury
48788,No Injury
98206,No Injury
29912,No Injury
147889,Moderate
117282,No Injury
13470,Minor
86043,No Injury
65121,No Injury


8.Reoder columns

In [22]:
column_order = [

    "crash_date",
    "traffic_control_device",
    "Weather",
    "Lighting",
    "Collision Type",
    "trafficway_type",
    "Road_alignment",
    "roadway_surface_cond",
    "road_defect",
    "crash_type",
    "Crash_severity",
    "Intersection_accident?",
    "Cost_of_damage",
    "prim_contributory_cause",
    "Number_of_vehicles_involved",
    "most_severe_injury",
    "Total_injuries_num",
    "Fatal_injuries_num",
    "injuries_incapacitating",
    "injuries_non_incapacitating",
    "injuries_not_visibily_evident",
    "injuries_no_indication",
    "crash_hour",
    "crash_day_of_week",
    "crash_month",
    "AM/PM"

]

Silver_df = Silver_df[column_order]

9.Adding conditional column: Time Buckets

In [23]:
conditions = [

    Silver_df["crash_hour"] <= 5,

    (Silver_df["crash_hour"] >= 6) &
    (Silver_df["crash_hour"] <= 11),

    (Silver_df["crash_hour"] >= 12) &
    (Silver_df["crash_hour"] <= 17),

    (Silver_df["crash_hour"] >= 18) &
    (Silver_df["crash_hour"] <= 21),

    (Silver_df["crash_hour"] >= 22) &
    (Silver_df["crash_hour"] <= 23)

]

values = [

    "Late Night",
    "Morning",
    "Afternoon",
    "Evening",
    "Night"

]

Silver_df["Time Buckets"] = np.select(
    conditions,
    values,
    default=None
)

10.Handling data inconsistency: trafficway_type cleanup

In [24]:
Silver_df["trafficway_type"] = (
    Silver_df["trafficway_type"]
    .replace({
        "UNKNOWN INTERSECTION TYPE":
        "UNKNOWN"
    })
)

11.Checking the Cost_of_damage column values

In [25]:
Silver_df['Cost_of_damage'].sample(10)

,Cost_of_damage
6963,"OVER $1,500"
171254,"OVER $1,500"
93562,"OVER $1,500"
201673,"OVER $1,500"
39307,"OVER $1,500"
40646,"$501 - $1,500"
153295,"$501 - $1,500"
14271,"$501 - $1,500"
191392,"OVER $1,500"
176599,"OVER $1,500"


Added new column damage_level that indicates the severity of damage 500 - => 1 , 501 - 1500 => 2 , 1500+ => 3

In [26]:
def damage_level(row):
    if row['Cost_of_damage'] == 'OVER $1,500':
     return 3
    elif row['Cost_of_damage'] == '$501 - $1,500':
      return 2
    elif row['Cost_of_damage'] == '$500 OR LESS':
      return 1
    else:
      return 'unknown'
Silver_df['Damage_level'] = Silver_df.apply(damage_level , axis = 1)

Check the new column (Damage_level)

In [27]:
Silver_df['Damage_level'].head(10)

,Damage_level
0,2
1,3
2,2
3,3
4,2
5,2
6,2
7,3
8,3
9,3


12.The data type of crash_date column is object

In [28]:
Silver_df['crash_date'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 209306 entries, 0 to 209305
Series name: crash_date
Non-Null Count   Dtype 
--------------   ----- 
209306 non-null  object
dtypes: object(1)
memory usage: 1.6+ MB


Changed the type of crash_date column to datetime

In [29]:
Silver_df['crash_date'] = pd.to_datetime(Silver_df['crash_date'])

Check the crash_date data type is changed successfully

In [30]:
Silver_df['crash_date'].head(5)

,crash_date
0,2023-07-29
1,2023-08-13
2,2021-12-09
3,2023-08-09
4,2023-08-19


### **Save Silver Layer**

In [31]:
Silver_df.to_csv(
    "/content/sample_data/Silver.csv",
    index=False
)

print("Silver Layer Created Successfully")

Silver Layer Created Successfully
